# Example 11.5.1 / 1: A logarithmic allocation model

Companion notebook for **11.5.1 Exponential Cones** in the AMPL lab Documentation, section "A logarithmic allocation model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A logarithmic allocation model

The same diminishing-returns structure from Nonlinear Models (AMPL lab documentation, section 11.2) can be recognized through exponential-cone structure by a capable MP driver and solver.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROJECTS;
param benefit {PROJECTS} > 0;
param budget > 0;

var Spend {PROJECTS} >= 0;
var Utility {PROJECTS};

maximize TotalUtility:
    sum {p in PROJECTS} benefit[p] * Utility[p];

subject to Budget:
    sum {p in PROJECTS} Spend[p] <= budget;

subject to LogHypograph {p in PROJECTS}:
    Utility[p] <= log(1 + Spend[p]);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROJECTS"] = ["A", "B"]
ampl.param["benefit"] = {"A": 2, "B": 1}
ampl.param["budget"] = 5

In [5]:
# run.py
ampl.solve(solver="gurobi", verbose=False)
print(ampl.solve_result)
print(ampl.var["Spend"].to_dict())
print(ampl.var["Utility"].to_dict())

solved
{'A': 3.66666665886644, 'B': 1.333333338800653}
{'A': 1.540445039275672, 'B': 0.8472978627303408}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROJECTS := A B;
param benefit := A 2 B 1;
param budget := 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver gurobi;
option solver_msg 0;
solve;
display Spend, Utility;

:    Spend    Utility     :=
A   3.66667   1.54045
B   1.33333   0.847298
;

